# astro-nbody — Quickstart

Run a small Plummer cluster, plot the energy drift, save to HDF5, reload.

**Pre-requisites:** kernel built (`make -C ../kernel`), package installed in `.venv` (`pip install -e ..[dev,notebooks]`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astro_nbody as nb

print(f'astro_nbody {nb.__version__}')
print(f'Available scenarios: {nb.list_scenarios()}')

## 1. Initialize a Plummer cluster

In [ ]:
sim = nb.Simulation(n=300, scenario='plummer', seed=42, softening=0.05)
print(f'N = {sim.n}')
print(f'Total mass = {sim.m.sum():.6f}')
print(f'Centre of mass = ({(sim.m*sim.x).sum():.2e}, {(sim.m*sim.y).sum():.2e}, {(sim.m*sim.z).sum():.2e})')

## 2. Plot the initial spatial distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].scatter(sim.x, sim.y, s=2, alpha=0.6, color='#60a5fa')
ax[0].set(xlabel='x', ylabel='y', title='xy projection', aspect='equal')
ax[1].hist(np.sqrt(sim.x**2 + sim.y**2 + sim.z**2), bins=30, color='#a78bfa', alpha=0.8)
ax[1].set(xlabel='radius r', ylabel='count', title='Radial distribution (Plummer profile)')
plt.tight_layout()
plt.show()

## 3. Integrate and track energy

In [ ]:
n_chunks = 50
steps_per_chunk = 20
dt = 0.01

times, K, U, E, Q = [], [], [], [], []

d = sim.diagnostics()
times.append(d.sim_time); K.append(d.K); U.append(d.U); E.append(d.E); Q.append(d.Q)

for _ in range(n_chunks):
    sim.run(steps=steps_per_chunk, dt=dt)
    d = sim.diagnostics()
    times.append(d.sim_time); K.append(d.K); U.append(d.U); E.append(d.E); Q.append(d.Q)

print(f'Final E = {E[-1]:+.6f}, |ΔE/E| = {abs((E[-1]-E[0])/E[0]):.2e}')
print(f'Final virial Q = {Q[-1]:.3f}')

## 4. Plot energy and virial vs time

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(times, K, color='#34d399', label='K')
ax[0].plot(times, U, color='#fbbf24', label='U')
ax[0].plot(times, E, color='#a78bfa', label='E', linewidth=2)
ax[0].axhline(E[0], color='#a78bfa', linestyle=':', alpha=0.4)
ax[0].set(ylabel='Energy (Hénon)', title='Energy components vs time')
ax[0].legend()
ax[1].plot(times, Q, color='#fb7185')
ax[1].axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax[1].set(xlabel='time (Hénon)', ylabel='2K/|U|', title='Virial ratio (1.0 = relaxed)')
plt.tight_layout(); plt.show()

## 5. Save and reload via HDF5

Files are interchangeable with what the Java service writes — same GADGET-like layout.

In [ ]:
out = sim.save('plummer_300_evolved.h5')
print(f'Wrote {out} ({out.stat().st_size:,} bytes)')

reloaded = nb.Simulation.from_hdf5(out)
print(f'Reloaded: N={reloaded.n}, sim_time={reloaded.sim_time:.3f}')
print(f'Position max diff = {np.max(np.abs(reloaded.x - sim.x)):.2e} (should be 0 — bit-exact)')